In [ ]:
from databricks.connect import DatabricksSession
from pyspark.sql.functions import (
    col,
    date_format,
    timestamp_diff,
    to_date,
    year,
    month,
    dayofmonth,
    dayofweek,
    hour,
    minute,
    second,
    quarter,
    weekofyear,
    unix_timestamp,
    date_trunc,
    round
)

spark = (DatabricksSession.builder
         .profile('vitalie')
         .serverless()
         .getOrCreate())

bronze_df = spark.table("workspace.bronze.nyctaxi_trips_raw")

silver_df = (bronze_df
                .select("trip_distance", 
                        "fare_amount", 
                        "pickup_zip", 
                        "dropoff_zip", 
                        "ingested_at", 
                        "_source_table",
                to_date("tpep_pickup_datetime").alias("pickup_date"),
                date_format("tpep_pickup_datetime", "yyyy").alias("pickup_year"),
                date_format("tpep_pickup_datetime", "MM").alias("pickup_month"),
                date_format("tpep_pickup_datetime", "dd").alias("pickup_day_of_month"),
                ((dayofweek("tpep_pickup_datetime")+5)%7+1).alias("pickup_day_of_week"),
                date_format("tpep_pickup_datetime", "EE").alias("pickup_day_name"),
                 round(
                        timestamp_diff(
                            "second",
                            "tpep_pickup_datetime",
                            "tpep_dropoff_datetime"
                        ) / 60,
                        2
        ).alias("trip_duration_minutes"),
                )
                # filter columns where rows have nulls
        .filter(("tpep_pickup_datetime is not null and "
                 "tpep_dropoff_datetime is not null and "
                 "pickup_zip is not null"))
                 #another method to filter null columns
        .filter((col("dropoff_zip").isNotNull()))
        .where(col("trip_distance") > 0)
        .where(col("trip_duration_minutes")>0)
        .withColumn("fare_per_mile", round(col("fare_amount")/col("trip_distance"), 2))
                )
display(silver_df)
target_table = "workspace.silver.nyctaxi_trips_clean"

try:
    silver_df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    row_count = spark.table(target_table).count()

    print(f"Table {target_table} saved succesfully with {row_count} rows")

except Exception as e:
    print(f"Failed to save table {target_table}")
    print(e)





,trip_distance,fare_amount,pickup_zip,dropoff_zip,ingested_at,_source_table,pickup_date,pickup_year,pickup_month,pickup_day_of_month,pickup_day_of_week,pickup_day_name,trip_duration_minutes,fare_per_mile
0,1.40,8.0,10103,10110,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-13,2016,02,13,6,Sat,9.37,5.71
1,1.31,7.5,10023,10023,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-13,2016,02,13,6,Sat,8.23,5.73
2,1.80,9.5,10001,10018,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-06,2016,02,06,6,Sat,11.57,5.28
3,2.30,11.5,10044,10111,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-12,2016,02,12,5,Fri,14.18,5.00
4,2.60,18.5,10199,10022,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-23,2016,02,23,2,Tue,30.62,7.12
5,1.40,6.5,10023,10069,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-13,2016,02,13,6,Sat,5.15,4.64
6,10.40,31.0,11371,10003,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-18,2016,02,18,4,Thu,23.00,2.98
7,10.15,28.5,11371,11201,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-18,2016,02,18,4,Thu,16.63,2.81
8,3.27,15.0,10014,10023,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-03,2016,02,03,3,Wed,19.27,4.59
9,4.42,15.0,10003,11222,2026-05-11 11:32:18.664543,samples.nyctaxi.trips,2016-02-19,2016,02,19,5,Fri,13.37,3.39


In [ ]:
# silver_df = (
#     bronze_df
#     .select(
#         # keep original timestamp columns
#         col("tpep_pickup_datetime"),
#         col("tpep_dropoff_datetime"),

#         # business columns
#         col("trip_distance"),
#         col("fare_amount"),
#         col("pickup_zip"),
#         col("dropoff_zip"),

#         # metadata columns
#         col("ingested_at"),
#         col("_source_table"),

#         # pickup date formats
#         to_date("tpep_pickup_datetime").alias("pickup_date"),
#         date_format("tpep_pickup_datetime", "yyyy-MM-dd").alias("pickup_format_yyyy_mm_dd"),
#         date_format("tpep_pickup_datetime", "dd-MM-yyyy").alias("pickup_format_dd_mm_yyyy"),
#         date_format("tpep_pickup_datetime", "MM/dd/yyyy").alias("pickup_format_mm_dd_yyyy"),
#         date_format("tpep_pickup_datetime", "yyyyMMdd").alias("pickup_format_yyyymmdd"),

#         # pickup datetime formats
#         date_format("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss").alias("pickup_datetime_24h"),
#         date_format("tpep_pickup_datetime", "yyyy-MM-dd hh:mm:ss a").alias("pickup_datetime_12h"),
#         date_format("tpep_pickup_datetime", "dd MMM yyyy HH:mm").alias("pickup_datetime_short_month"),
#         date_format("tpep_pickup_datetime", "dd MMMM yyyy HH:mm:ss").alias("pickup_datetime_full_month"),
#         date_format("tpep_pickup_datetime", "EEEE, dd MMMM yyyy").alias("pickup_full_day_name"),

#         # pickup time only
#         date_format("tpep_pickup_datetime", "HH:mm:ss").alias("pickup_time_24h"),
#         date_format("tpep_pickup_datetime", "hh:mm:ss a").alias("pickup_time_12h"),

#         # pickup date parts
#         year("tpep_pickup_datetime").alias("pickup_year"),
#         quarter("tpep_pickup_datetime").alias("pickup_quarter"),
#         month("tpep_pickup_datetime").alias("pickup_month"),
#         dayofmonth("tpep_pickup_datetime").alias("pickup_day"),
#         dayofweek("tpep_pickup_datetime").alias("pickup_day_of_week"),
#         weekofyear("tpep_pickup_datetime").alias("pickup_week_of_year"),
#         hour("tpep_pickup_datetime").alias("pickup_hour"),
#         minute("tpep_pickup_datetime").alias("pickup_minute"),
#         second("tpep_pickup_datetime").alias("pickup_second"),

#         # unix timestamp
#         unix_timestamp("tpep_pickup_datetime").alias("pickup_unix_timestamp"),

#         # truncated timestamps
#         date_trunc("DAY", col("tpep_pickup_datetime")).alias("pickup_trunc_day"),
#         date_trunc("MONTH", col("tpep_pickup_datetime")).alias("pickup_trunc_month"),
#         date_trunc("YEAR", col("tpep_pickup_datetime")).alias("pickup_trunc_year"),

#         # dropoff examples
#         to_date("tpep_dropoff_datetime").alias("dropoff_date"),
#         date_format("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss").alias("dropoff_datetime_24h"),
#         date_format("tpep_dropoff_datetime", "dd-MM-yyyy HH:mm:ss").alias("dropoff_datetime_eu"),
#         date_format("tpep_dropoff_datetime", "MM/dd/yyyy hh:mm:ss a").alias("dropoff_datetime_us_12h"),

#         # simple calculated column
#         round(
#             (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,
#             2
#         ).alias("trip_duration_minutes")
#     )
#     .where(col("trip_distance") > 0)
#     .where(col("fare_amount") >= 0)
# )

# silver_df.write \
#     .mode("overwrite") \
#     .format("delta") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("workspace.silver.nyctaxi_trips_clean")